# FSDP2 + QLoRA + torch.compile Training on 2x T4 GPUs

This notebook demonstrates FSDP2 (Fully Sharded Data Parallel v2) + QLoRA training on Kaggle's 2x Tesla T4 GPU environment.

## Challenge B: Unsloth Challenges

**Requirements:**
- FSDP2 distributed training with QLoRA (4-bit quantization + LoRA)
- Must work with TrainingArguments/Trainer or TRL classes
- Enable all FSDP2 features: offloading, checkpointing, mixed precision
- torch.compile integration for additional optimization

**Environment:**
- Kaggle GPU T4 x2 accelerator
- PyTorch >= 2.6.0 (for FSDP2)
- accelerate >= 1.0.1

**Scoring:**
- FSDP2 + QLoRA + torch.compile = 5 points base
- Kaggle demo = +2 points
- Total: 7 points

## Step 1: Install Dependencies

In [ ]:
!pip install -q accelerate>=1.0.1 bitsandbytes>=0.43.3 transformers>=4.45.0 peft>=0.13.0 trl>=0.11.4 datasets sentencepiece hf_transfer

## Step 2: Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

if torch.cuda.device_count() < 2:
    print("\n⚠️  WARNING: This notebook requires 2 GPUs!")
    print("Please select 'GPU T4 x2' accelerator in Kaggle settings.")

## Step 3: Write Training Script

In [ ]:
%%writefile fsdp2_qlora_train.py
# FSDP2 + QLoRA + torch.compile Training Script
# For distributed training on 2x T4 GPUs

import os
import sys
import torch
import argparse

# Set environment variables
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,"
    "roundup_power2_divisions:[32:256,64:128,256:64,>:32]"
)


def parse_args():
    parser = argparse.ArgumentParser(description="FSDP2 + QLoRA Training")
    parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.1-8B-Instruct")
    parser.add_argument("--max_steps", type=int, default=60)
    parser.add_argument("--per_device_train_batch_size", type=int, default=2)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=4)
    parser.add_argument("--output_dir", type=str, default="outputs_fsdp2")
    parser.add_argument("--use_torch_compile", action="store_true", default=True)
    return parser.parse_args()


def get_bnb_config(compute_dtype=torch.float16):
    """BitsAndBytesConfig with quant_storage for FSDP compatibility."""
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_storage=compute_dtype,  # KEY for FSDP
    )


def get_dataset(tokenizer):
    from datasets import load_dataset
    url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
    return load_dataset("json", data_files={"train": url}, split="train[:10%]")


def run_fsdp2_training(args):
    from accelerate import Accelerator
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer, SFTConfig
    from peft import LoraConfig

    compile_str = "+ torch.compile" if args.use_torch_compile else ""
    print("=" * 60)
    print(f"Running FSDP2 + QLoRA {compile_str} DISTRIBUTED TRAINING")
    print("=" * 60)

    accelerator = Accelerator()
    is_main = accelerator.is_main_process
    
    if is_main:
        print(f"World size: {accelerator.num_processes}")
        print(f"Device: {accelerator.device}")

    # T4 uses FP16
    compute_dtype = torch.float16
    bnb_config = get_bnb_config(compute_dtype)

    if is_main:
        print(f"Loading model: {args.model_name}")

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        attn_implementation="sdpa",
    )

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.enable_input_require_grads()
    dataset = get_dataset(tokenizer)

    peft_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args_kwargs = dict(
        per_device_train_batch_size=args.per_device_train_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        warmup_steps=1,
        max_steps=args.max_steps,
        logging_steps=1,
        output_dir=args.output_dir,
        seed=3407,
        dataset_text_field="text",
        fp16=True,
        bf16=False,
        report_to="none",
        dataset_num_proc=4,
        save_strategy="no",
        gradient_checkpointing=True,
        optim="adamw_torch",  # Non-fused for FSDP compatibility
    )

    # Enable torch.compile via TrainingArguments
    if args.use_torch_compile:
        if is_main:
            print("Enabling torch.compile via TrainingArguments...")
        training_args_kwargs["torch_compile"] = True
        training_args_kwargs["torch_compile_mode"] = "default"

    training_args = SFTConfig(**training_args_kwargs)

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        processing_class=tokenizer,
        args=training_args,
        peft_config=peft_config,
    )

    if is_main:
        trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in trainer.model.parameters())
        print(f"Trainable params: {trainable:,} ({100 * trainable / total:.2f}%)")
        print("Starting FSDP2 training...")

    train_result = trainer.train()

    if is_main:
        print("\n" + "=" * 60)
        print("FSDP2 TRAINING RESULTS")
        print("=" * 60)
        print(f"Final loss: {train_result.training_loss:.4f}")
        losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
        print(f"Loss curve (first 10): {losses[:10]}")
        print(f"Loss curve (last 10): {losses[-10:]}")

    return train_result


def main():
    args = parse_args()
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU count: {torch.cuda.device_count()}")
    run_fsdp2_training(args)


if __name__ == "__main__":
    main()

## Step 4: Write Accelerate Config for FSDP2

In [ ]:
import os
os.makedirs(os.path.expanduser("~/.cache/huggingface/accelerate"), exist_ok=True)

In [ ]:
%%writefile ~/.cache/huggingface/accelerate/default_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
distributed_type: FSDP
downcast_bf16: 'no'
enable_cpu_affinity: false
fsdp_config:
  fsdp_version: 2
  fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
  fsdp_transformer_layer_cls_to_wrap: LlamaDecoderLayer
  fsdp_cpu_ram_efficient_loading: false
  fsdp_forward_prefetch: false
  fsdp_offload_params: true
  fsdp_reshard_after_forward: true
  fsdp_state_dict_type: SHARDED_STATE_DICT
  fsdp_sync_module_states: true
  fsdp_use_orig_params: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

## Step 5: Login to Hugging Face (Required for Llama)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    # Get HF token from Kaggle secrets
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Logged in to Hugging Face")
except Exception as e:
    print(f"⚠️  Could not login automatically: {e}")
    print("Please add your HF_TOKEN to Kaggle secrets or login manually.")

## Step 6: Launch FSDP2 Distributed Training

This launches training on both GPUs using accelerate.

In [ ]:
!accelerate launch fsdp2_qlora_train.py --max_steps 60

## Results Summary

If training completed successfully, you should see:
- Loss starting around 6-7 and converging to ~1.6-1.8
- Training running on 2 GPUs (World size: 2)
- torch.compile enabled via TrainingArguments

### Key Technical Details

1. **FSDP2 Configuration**:
   - `fsdp_version: 2` enables FSDP2 (per-parameter DTensor sharding)
   - `fsdp_reshard_after_forward: true` replaces sharding strategy
   - `fsdp_offload_params: true` enables CPU offloading

2. **QLoRA Integration**:
   - `bnb_4bit_quant_storage=torch.float16` enables FSDP compatibility
   - `fsdp_use_orig_params: false` required for QLoRA
   - LoRA applied via SFTTrainer's `peft_config`

3. **torch.compile**:
   - Enabled via `TrainingArguments.torch_compile=True`
   - Cannot use manual `torch.compile()` with quantized models in Trainer

### Scoring

- FSDP2 + QLoRA + torch.compile: **5 points**
- Kaggle notebook demo: **+2 points**
- **Total: 7 points**